<div class="alert alert-info">
    
➡️ Before you start, make sure that you are familiar with the **[study guide](https://liu-nlp.ai/text-mining/logistics/)**, in particular the rules around **cheating and plagiarism**.

➡️ Make sure you fill in **all** cells that say **`YOUR CODE HERE`** or **YOUR ANSWER HERE**.  You normally shouldn't need to modify any of the other cells.

➡️ Make sure that you save the notebook **with the outputs from all cells visible,** as if you had run the notebook from top to bottom.

</div>

# L1: Information Retrieval

<div class="alert alert-warning">

⚠️ Parts of this lab **require a GPU with 8GB+ VRAM.**  Unless you have such a GPU, you will likely not be able to run this part of the lab.  Remember that you can access the GPU-equipped lab rooms in B-huset – Åsgard, Egypten, Olympen – to work on this lab, preferably during the times we have booked one or more of these rooms for the course.

</div>


In this lab, you will use techniques from information retrieval to implement the core of a minimalistic search engine. The data for this lab consists of a collection of app descriptions scraped from the [Google Play Store](https://play.google.com/store/apps?hl=en). From this collection, your search engine should retrieve those apps whose descriptions best match a given query under the vector space model.

In [1]:
import torch
from liu_tm import check_solutions

## Dataset

The app descriptions come in the form of a compressed [JSON](https://en.wikipedia.org/wiki/JSON) file. Start by loading this file into a [Pandas DataFrame](https://pandas.pydata.org/docs/user_guide/dsintro.html#dataframe).

In [2]:
import bz2
import numpy as np
import pandas as pd
pd.set_option('display.max_colwidth', 500)

with bz2.open('app-descriptions.json.bz2', mode='rt', encoding='utf-8') as source:
    df = pd.read_json(source, encoding='utf-8')

In Pandas, a DataFrame is a table with indexed rows and labelled columns of potentially different types. You can access data in a DataFrame in various ways, including by row and column. To give an example, the code in the next cell shows rows 200–204:

In [3]:
df.loc[200:205]

,name,description
200,Brick Breaker Star: Space King,"Introducing the best Brick Breaker game that everyone can enjoy.\nEnjoy various missions and addictively simple play control.\n\n[Features]\n- Hundreds of stages and various missions\n- No limit to play such as Heart, play as much as you can!\n- 5 kinds of various items and items reinforcement system\n- No network required\n- game file is as low as 20M, light-weight download!\n- supports tablet screen\n- supports Google Play Leaderboards, Achievement, Multiplay\n- supports 14 languages\n\nHo..."
201,Brick Classic - Brick Game,"Classic Brick Game!\n\nBrick Classic is a popular and addictive puzzle game!\n\nHow to play?\n- Simply drag the bricks to move them.\n- Create full lines on the grid vertically or horizontally to break bricks.\n\nTips:\n- Classic brick game without time limits.\n- Place the bricks in a reasonable position.\n- The more brick break, the more scores you have.\n- Bricks can't be rotated.\n\nWho's the best brick breaker? Challenge it now!!!"
202,Bricks Breaker - Glow Balls,"Bricks Breaker - Glow Balls is a addictive and challenging brick game.\nJust play it to relax your brain. Be focus on breaking bricks and you will find it more funny and exciting.\n\nHow to play\n- Hold the screen with your finger and move to aim.\n- Find best positions and angles to hit all bricks.\n- When the durability of brick reaches 0, destroyed.\n- Never let bricks reach the bottom or game is over.\n\nFeatures\n- Colorful glow skins.\n- Free to play.\n- Easy game controls with one fin..."
203,Bricks Breaker Quest,"How to play\n- The ball flies to wherever you touched.\n- Clear the stages by removing bricks on the board.\n- Break the bricks and never let them hit the bottom.\n- Find best positions and angles to hit every brick.\n\nFeature\n- Free to play\n- Tons of stages\n- Various types of balls\n- Easy to play, Simplest game system, Designed for one handheld gameplay.\n- Off-line (without internet connection) gameplay supported \n- Multi-play supported\n- Tablet device supported\n- Achievement & lea..."
204,Brothers in Arms® 3,"Fight brave soldiers from around the globe on the frenzied multiplayer battlegrounds of World War 2 or become Sergeant Wright and experience a dramatic, life-changing single-player journey, in the aftermath of the D-Day invasion.\n\nCLIMB THE ARMY RANKS IN MULTIPLAYER \n> 4 maps to master and enjoy. \n> 2 gameplay modes to begin with: Free For All and Team Deathmatch.\n> Unlock game-changing perks by playing with each weapon class!\n> A soldier’s only as deadly as his weapon. Be sure to upgr..."
205,Brown Dust - Tactical RPG,"The Empire has fallen, and the Age of Great Mercenaries Now Begins!\nCreate Your Ultimate Team And Strike Down Your Enemies!\n\nCAPTIVATING AND STUNNING ARTWORK\n- Experience the high-quality anime illustrations you have never seen before.\n- Meet Brown Dust's charming Mercenaries now.\n\nASSEMBLE LEGENDARY MERCENARIES\n- Over 300 Mercenaries and a Variety of Skills.\n- Discover the Unique Mercenaries, 6 Devils and Dominus Octo.\n- All Mercenaries can reach max level and the highest rank.\n\..."


As you can see, there are two labelled columns: `name` (the name of the app) and `description` (a textual description). The next cell shows how to access only the description field from row 200:

In [4]:
df.loc[200, 'description']

'Introducing the best Brick Breaker game that everyone can enjoy.\nEnjoy various missions and addictively simple play control.\n\n[Features]\n- Hundreds of stages and various missions\n- No limit to play such as Heart, play as much as you can!\n- 5 kinds of various items and items reinforcement system\n- No network required\n- game file is as low as 20M, light-weight download!\n- supports tablet screen\n- supports Google Play Leaderboards, Achievement, Multiplay\n- supports 14 languages\n\nHomepage:\nhttps://play.google.com/store/apps/dev?id=4931745640662708567\n\nFacebook: \nhttps://www.facebook.com/spcomesgames/'

## Problem 1: TF–IDF vectors

We start by vectorising the data — more specifically, we map each app description to a tf–idf vector. This is very simple with a library like [scikit-learn](https://scikit-learn.org/stable/), which provides a [TfidfVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html) class for exactly this purpose.  If we instantiate this class, and call `fit_transform()` on all of our app descriptions, scikit-learn will preprocess and tokenize each app description, compute tf–idf values for each of them, and return a vectorised representation:

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df['description'])
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 267110 stored elements and shape (1614, 27877)>

Let’s pick the app "Pancake Tower", which has a rather short description text, to see how it has been vectorised:

In [6]:
# We can use 'toarray' to convert the sparse matrix object into a "normal" array
vec = X[1032].toarray()[0]

# The app description & its corresponding vector
df.loc[1032, 'description'], vec

("Let's see how many pancakes you can pile up!!",
 array([0., 0., 0., ..., 0., 0., 0.], shape=(27877,)))

That's not very informative yet.  We know that the vector contains tf–idf values, and that each dimension of the vector corresponds to a token in the vectorizer’s vocabulary; let's extract these for this specific example.

<div class="alert alert-info">

Your first task is to **display all the tokens** that are represented in the vector of the "Pancake Tower" app **with a tf–idf value greater than zero** (i.e., only the tokens that are actually part of this app’s description), **in descending order** of the tf–idf values.  In other words, the token with the highest tf–idf value should be at the top of your output, and the token with the lowest tf–idf value at the bottom.   Before you implement this, think about what you would expect the output look like, for example which words you would expect to have the highest/lowest tf–idf values in this example.

</div>

_Hints:_ You can check [the documentation of `TfidfVectorizer`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html) to find out how to access the `vectorizer`’s vocabulary.

In [7]:
"""Print the tokens and their tf–idf values, in descending order."""
vocab = vectorizer.get_feature_names_out()
inds = np.where(vec > 0)[0]
tfidf_1032 = dict(zip(vocab[inds], vec[inds]))

sorted_tfidf_1032 = dict(sorted(tfidf_1032.items(), key=lambda i: i[1], reverse=True))
sorted_tfidf_1032

{'pancakes': np.float64(0.6539332651185913),
 'pile': np.float64(0.5304701435508047),
 'let': np.float64(0.2615287714771797),
 'see': np.float64(0.2557630827415271),
 'many': np.float64(0.23491959669849022),
 'how': np.float64(0.21153246225085887),
 'up': np.float64(0.17216837691451817),
 'can': np.float64(0.13047602895910532),
 'you': np.float64(0.10276923239718011)}

## Problem 2: Keyword extraction with TF–IDF

There are a few tasks for which tf–idf vectors are especially suitable.  One such task is extracting salient (_i.e.,_ important, informative) keywords from a document.  A simple method to do this is to simply pick the top-$n$ terms with the highest tf–idf value, still using the vectorizer we fitted in Problem&nbsp;1.

<div class="alert alert-info">

Implement a function that **extracts the top-$n$ keywords** based on tf–idf scores from a given document.

</div>


In [8]:
new_text = "You are a penis with dildo"
new_vector = vectorizer.transform([new_text])
new_vector[0].toarray()[0]

array([0., 0., 0., ..., 0., 0., 0.], shape=(27877,))

In [9]:
def keywords(text, n=10):
    """
    Arguments:
      text (str): The text from which to extract keywords.
      n (int): The number of keywords to extract. [default: 10]

    Returns:
      A list containing the `n` most salient keywords from `text`, as measured by
      their tf–idf value relative to the collection of app descriptions.
    """
    text_vector = vectorizer.transform([text])[0].toarray()[0]

    # Get the indeces of tf-idfs scores > 0:
    inds = np.where(text_vector > 0)[0]
    tfidf = dict(zip(vocab[inds], text_vector[inds]))
    # Sort the dictionary in descending order:
    sorted_tfidf = dict(sorted(tfidf.items(), key=lambda i: i[1], reverse=True))

    return list(sorted_tfidf.keys())[:n]

### 🤞 Test your code

Test your implementation by running the following cell, which uses the app description of "Train Conductor World".  Read the description first to guess what you would expect to be the most salient keywords here!

In [10]:
print(df.iloc[1428]["description"])

Master and manage the chaos of international railway traffic as the ultimate railroad tycoon. Build the rail network of your dreams; lay rails and solve the railroad puzzle with branching and forking roads at every turn. Become the richest manager and pick your path, do you optimise to the micro level, planning routes and managing the timetable, or sit idle letting your business keep earning while you sleep! 

Get in the driver's seat and take passengers to their destinations, dropping them at stations, and haul goods to ports and factories. Control and conduct the trains, marshalling them through tunnels, around obstacles and over mountains in this thrilling, fast-paced action arcade videogame. Connect your express trains across the railyard at breakneck speed. Try to avoid crashing, but I bet you can’t. You’ll need snappy strategy to master the chaos! Be on high-alert for explosive crashes, near-misses and split second situations.

Ring the bell and toot your horn while playing in al

In [11]:
"""Extract the most salient keywords from the description of 'Train Conductor World'."""

out = keywords(df.iloc[1428]["description"])
display(out)

with check_solutions() as check:
    check.equals(len(out), 10, "The function should return exactly 10 results")
    check("trains" in out, "The token 'trains' should be among the top 10")

['trains',
 'the',
 'and',
 'railroad',
 'railway',
 'chaos',
 'your',
 'forking',
 'haul',
 'locomotives']

Before you continue, think about the results that you got.  Are they as you would have expected?  Did any of the keywords surprise you?

## Problem 3: Finding the nearest vectors

To build a small search engine, we need to be able to turn _queries_ (for example the string "pile up pancakes") into _query vectors_, and then find out which of our app description vectors are closest to the query vector.

For the first part (turning queries into query vectors), we can simply re-use the `vectorizer` that we used for the app descriptions. For the second part, an easy way to find the closest vectors is to use scikit-learn’s [NearestNeighbors](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.NearestNeighbors.html) class. This class needs to be _fit_ on a set of vectors (the "training set"; in our case the app descriptions) and can then be used with any vector to find its _nearest neighbors_ in the vector space.

<div class="alert alert-info">

Instantiate and fit a class that returns the _ten (10)_ nearest neighbors!

</div>


In [12]:
from sklearn.neighbors import NearestNeighbors

"""Instantiate and fit a class that returns the 10 nearest neighboring vectors."""
neigh = NearestNeighbors(n_neighbors=10, metric='cosine')
neigh.fit(X)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",10
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
Name,Type,Value
effective_metric_ effective_metric_: strMetric used to compute distances to neighbors.,str,'cosine'
effective_metric_params_ effective_metric_params_: dictParameters for the metric used to compute distances to neighbors.,dict,{}


<div class="alert alert-info">

Next, implement a function that uses the vectorizer and the fitted class to find the nearest neighbours for a given query string.

</div>

In [13]:
def search(query):
    """Find the nearest neighbors in `df` for a query string.

    Arguments:
      query (str): A query string.

    Returns:
      The 10 apps (with name and description) most similar (in terms of
      cosine similarity) to the given query as a Pandas DataFrame.
    """

    query_vec = vectorizer.transform([query])[0].toarray()[0]

    # Find 10 NN:
    neighs = neigh.kneighbors([query_vec], return_distance=False).flatten()

    return df.iloc[neighs]

### 🤞 Test your code

Test your implementation by running the following cell, which will sanity-check your return value and show the 10 best search results for the query _"pile up pancakes"_:

In [14]:
"""Check that searching for "pile up pancakes" returns a DataFrame with ten results,
   and that the top result is "Pancake Tower"."""

result = search('pile up pancakes')
display(result)

with check_solutions() as check:
    check(isinstance(result, pd.DataFrame), "search() function should return a Pandas DataFrame")
    check.equals(len(result), 10, "search() function should return 10 search results")
    check.equals(result.iloc[0]["name"], "Pancake Tower", "The top search result should be 'Pancake Tower'")

,name,description
1032,Pancake Tower,Let's see how many pancakes you can pile up!!
326,Cooking School: Games for Girls,"Children like to help their parents. They especially like to help with cooking . When there is a cooking in the kitchen, it is no way to play. But cooking is a complicated process and often it ends up with a huge mess in the kitchen. But what if you are so eager to cook pancakes, cake or cupcakes? How to cook all that without doing a cleaning after? We have a solution! Home Cooking School with our curious Hippo has opened especially for parents and children! We do not only cook food here. We..."
656,"Hell’s Cooking — crazy chef burger, kitchen fever","⭐ ⭐ ⭐ ⭐ ⭐ New world of crazy cooking is here. Feel what it means to be a master chef who prepares fantastic fast food in a prominent king kitchen! If you haven't ever tried yourself as a hamburger chef cook, it's possibly the best time for making diner. Download and launch Hell's Cooking — crazy chef burger, kitchen fever HD game and get prepared to jump into a fever and adventurous perfect world of burgers.\n\nNew girls game Hell's Cooking gives you lots of opportunities for your crazy cafe..."
1235,Solitaire,"Solitaire Free by Solitaire Card Games is the #1 klondike solitaire games on android. The solitaire Free is popular and classic card games you know and love.\n\nWe carefully designed a fresh solitaire free modern look, woven into the wonderful solitaire classic feel that everyone loves. \n\nExperience the crisp, clear, and easy to read cards, simple and quick animations, and subtle sounds, in either landscape or portrait views. \n\nYou can move cards with a single tap or drag them to their d..."
1164,Rummy - Free,"Play the famous Rummy card game on your Android Smartphone or Tablet !! \n\nPlay rummy with 2, 3, or 4 players against simulated opponents playing with high-level artificial intelligence. \nThere are a number of rules that can be modified, making this game very faithful to the original. \n\n*** MANY VARIATIONS INCLUDED *** \n\nMany rummy variations are included in the application: \n\n- From 2 to 4 players. \n- Choose the AI level of opponents. \n- Number of cards dealt to each player (from ..."
1181,Sago Mini Trucks and Diggers,"Drive a dump truck with Rosie the hamster! Pile dirt high and dig deep in the ground with diggers, cranes and bulldozers. Build a home for a new friend! Choose a barn, a castle or even a cupcake-house. Don’t forget to add the finishing touches for the proud owner.\n\nOn this construction site, kids love being the boss. With six mighty machines and piles of dirt, you can build all day! Part of the award-winning suite of Sago Mini apps, this app puts kids in charge.\n\nSago Mini apps have no i..."
436,Dr. Panda's Ice Cream Truck,"Chocolate? Vanilla? Strawberry? All three!? You decide! In Dr. Panda’s Ice Cream Truck you can mix up all sorts of different flavors with cookies, chocolate, nuts and more to make the perfect ice cream—hundreds of combinations in all.\n\nScoop it!\nThese animals love ice cream, and will eat as much (or little) as you want to serve them. You can make scoops big or small and pile them as high as you want—using any of the ice cream you’ve created!\n\nToppings galore!\nUse chocolate syrup, cooki..."
1442,Turbo Dismount™,"The legendary crash simulator is now on Google Play!\n\nPerform death-defying motor stunts, crash into walls, create traffic pile-ups of epic scale - and share the fun!\n\nTurbo Dismount™ is a kinetic tragedy about Mr. Dismount and the cars who love him. It is the official sequel to the wildly popular and immensely successful personal impact simulator - Stair Dismount™. \n\nFEATURES:\n* Flinch-inducing crash physics\n* Crunchy sound effects\n* Delicious slow-mo replay system\n* Multiple vehi..."
1446,UNO!™,"Play the world’s number one card game like never before. UNO!™ has all-new rules, tournaments, adventures and so much more! At home or on the move, jump into games instantly. Whet

Before continuing with the next problem, play around a bit with this simple search functionality by trying out different search queries, and see if the results look like what you would expect:

In [15]:
# Example — try out your own queries!
search("Universe")

,name,description
1192,Second Galaxy,"[Follow Us]\n\nFacebook: https://www.facebook.com/secondgalaxy/\nInstagram: https://www.instagram.com/secondgalaxy_global/\nOfficial Website: https://secondgalaxy.zlongame.com\n\nSecond Galaxy presents a vast open sci-fi world combining RPG and SLG elements and consisting of 4,961 galaxies, each filled with all manner of cosmic dust, dark matter, wormholes, space stations, and other celestial bodies and artificial space facilities. In the process of exploring the game, you can command all ki..."
1307,Super Brawl Universe,"Join iconic characters from Nickelodeon’s all-time, most popular shows to battle it out in the ultimate arcade action game! \n\nUnleash awesome super moves with your favorite Nick characters: SpongeBob, Invader Zim, Kid Danger, Danny Phantom, and lots more! Battle it out at the Krusty Krab, the Loud House, the Avatar Air Temple, and all the other incredible locations. \n\nEnter the Super Brawl tournament, assemble the ultimate team, and fight your way to the top!\n\nBUILD YOUR DREAM TEAM\n- ..."
1446,UNO!™,"Play the world’s number one card game like never before. UNO!™ has all-new rules, tournaments, adventures and so much more! At home or on the move, jump into games instantly. Whether an UNO!™ veteran or completely new, take on challenges and reap the rewards. UNO!™ is the ultimate competitive family-friendly card game.\n- Play classic UNO!™ or use tons of popular house rules!\n- Connect anytime, anywhere with friends from around the world! \n- Two heads are better than one in 2v2 mode. Use t..."
588,Galaxy Glory,"Galaxy Glory is a game of the Galaxy space shooting type.\nIf you like to simulate a starry sky, then Galaxy Glory is definitely your best choice because it is perfect for you.\n\nIn the future expansion of human civilization, suffering from the inexplicable alien attack in the depths🛸 of the universe, the real universe starry sky team began,\nAs an experienced pilot, you are called by the state to protect the Milky Way from alien invaders. In order to complete this task,You must show your c..."
849,MARVEL Contest of Champions,"Prepare for epic versus-fighting action with your favorite Marvel Super Heroes & Super Villains in the ultimate cosmic showdown! Spider-Man, Iron Man, Wolverine & more await your summons to battle! Assemble a team & begin your quest to become the Ultimate Marvel Champion!\n\nWELCOME TO THE CONTEST:\nCaptain America vs. Iron Man! Hulk vs. Wolverine! Spider-Man vs. Deadpool! The greatest battles in Marvel history are in your hands! The greedy Elder of the Universe known as The Collector has su..."
775,KleptoCats Cartoon Network,"The KleptoCats have returned to invade the colorful worlds of popular Cartoon Network series such as Adventure Time, The Amazing World of Gumball, The Powerpuff Girls, Steven Universe and We Bare Bears.\n\nSome have theorized that we are not alone in the multiverse… that there are creatures out there far more clever, more smarter, more advanced-er and more cuter-er than we could possibly fathom. They were right! Blossom, Ice Bear, Steven, Gumball and Finn are about to learn the hard way that..."
351,DC Legends: Battle for Justice,"YOU CAN’T SAVE THE WORLD ALONE - Join your favorite Justice League heroes and villains in epic RPG combat. \n\nRecruit Justice League heroes like Batman, Superman, The Flash, Wonder Woman and more in the ultimate DC experience. Join dynamic PvP battles and form alliances to clash against rival teams as you fight your way across the galaxy. Empower your heroes with epic super power attacks, improve their stats and upgrade gear as you gain experience. The future of your team is in your hand - ..."
1391,Time Surfer,"50% off all purchases for a limited time!!!\n\nSurf for your life across the galaxy! An inevitable destructive force is after you... The end of the universe! Your only hope? You can cheat death by rewinding time! Get ready for the most adrenaline-pumping surf of the year!\n\nPRE

## Problem 4: Dense embedding models

Large language models, based on neural network architectures, also allow us to turn documents into vectors, and there are even specific embedding models that are optimized for this purpose.  A modern one is [EmbeddingGemma2](https://developers.googleblog.com/embeddinggemma-2-the-developer-guide/), trained by Google DeepMind.  We will use this model to implement the same search functionality as with the tf–idf vectors above.

In [16]:
from pathlib import Path
from liu_tm.torch import check_gpu

check_gpu()
# This part checks if we’re on the lab computers, and loads the model from there
model_name = Path("/courses/TDDE16/models/google/embeddinggemma-2")
if not model_name.exists():
    model_name = "google/embeddinggemma-2"

An embedding model like EmbeddingGemma2 is best used with the [SentenceTransformers](https://www.sbert.net/) library. Let's load the model first. If you are **not** running this on the lab computers, the first time you run this, it will **automatically download the `google/embeddinggemma-2` model** (ca. 1.5 GB of data).

Note that EmbeddingGemma2 is actually _multimodal_, meaning it can work not only with text, but even audio, images, and video. We only need the text capacity here, so we specify that in the config to save some GPU memory:

In [17]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    model_name,
    # Text only (270M parameters)
    config_kwargs={"vision_config": None, "audio_config": None},
)

W1108 14:49:22.894000 8736 course-materials/.venv/lib/python3.13/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/413 [00:00<?, ?it/s]

The following cell shows how to use this model, adapted directly from [EmbeddingGemma2’s developer guide](https://developers.googleblog.com/embeddinggemma-2-the-developer-guide/).  `model.encode()` returns the embedding vectors, while `model.similarity()` computes their cosine similarity:

In [18]:
query = "pile up pancakes"
document = "Let's see how many pancakes you can pile up!!"

query_emb = model.encode(query, prompt_name="SearchQuery")
doc_emb = model.encode(document, prompt_name="Document")
print(model.similarity(query_emb, doc_emb))

tensor([[0.8748]])


Let's compute document embeddings for our entire dataset next.

In [19]:
# If GPU memory becomes an issue, uncommenting the following line can help –
# it cuts off very long documents (should only affect ~32 documents in our data)
# model.max_seq_length = 1024

doc_embs = model.encode(
    df['description'].tolist(),
    prompt_name="Document",
    # Batch size is another parameter that can be tweaked –
    # increasing batch size uses more GPU memory, but is faster,
    # decreasing batch size uses less GPU memory, but is slower.
    batch_size=32,
    show_progress_bar=True,
)
doc_embs.shape

Batches:   0%|          | 0/51 [00:00<?, ?it/s]

(1614, 768)

In [20]:

sim_tens = model.similarity(query_emb, doc_embs)
_, top_inds = torch.topk(sim_tens, 10)
df.iloc[top_inds.flatten()]

,name,description
1032,Pancake Tower,Let's see how many pancakes you can pile up!!
1185,Sandwich!,"Make delicious sandwiches by folding toppings on top of each other, and serve it on a platter to eat!\n\nIn this new, delicious puzzle game it’s your job to combine bread with lettuce, tomato, salmon, cheese, onion, and many more toppings. Unlock more ingredients as you go along and the levels get harder! Are you a true sandwich master?\n\nFull of tricky levels, this game is set to eradicate your boredom. Oddly satisfying and deliciously entertaining! How high can you stack your sandwich? Re..."
343,Crowd City,Become the biggest crowd in town!\nGather people accross the city and crush your opponents with your overwhelming leadership!
544,Flip Dunk,Tap to jump and then hold your finger while in the air to flip.
175,Block Puzzle Infinite,"Drag the blocks on the bottom board to the checkerboard, try to make more full rows or columns to get score, you will lose the game if no spaces to put the whole block, having fun!"
975,My Virtual Love,New stunning application!
465,Drop Stack Ball - Fall Helix Blast Crash 3D,"The most addictive stack fall game with optimised performance for all devices!\nHelix Stack Ball is a super fun and addictive one touch casual game.\nPress and hold on the screen and let the ball go down without touching the obstacles! Hold as long as possible to make combo and break the black blocks. Let the ball fall down from the helix stacks.\nStack Ball is a 3d arcade game where players smash, bump and bounce through revolving helix platforms to reach the end.\nStack Blast Ball is a Bra..."
676,Hoop Stars,Dunk a lot! You have never played sport like this before! Feel the fun of true “reverse” basketball dribbling. Challenge the world and collect all the trophies!\nDunk a lot! You have never played sport like this before! Feel the fun of true “reverse” basketball dribbling. Challenge the world and collect all the trophies!
326,Cooking School: Games for Girls,"Children like to help their parents. They especially like to help with cooking . When there is a cooking in the kitchen, it is no way to play. But cooking is a complicated process and often it ends up with a huge mess in the kitchen. But what if you are so eager to cook pancakes, cake or cupcakes? How to cook all that without doing a cleaning after? We have a solution! Home Cooking School with our curious Hippo has opened especially for parents and children! We do not only cook food here. We..."
391,Dinosaur Rampage,Dinosaurs have been resurrected in the present age!\n\nMove your dinosaur with simple controls. Win by eating the most people and causing the most destruction! Collect coins and make your dinosaur stronger!\n\nCan you destroy the highest tower?


We have everything we need now to recreate the same search functionality that we wrote in Problem 3.

<div class="alert alert-info">

Implement a function that uses the embedding model and the computed document embeddings to find the nearest neighbours for a given query string.

</div>

_Hint:_ You don’t need to use scikit-learn for that; SentenceTransformers and Torch tensors come with all the functionality we need for this.  In particular, you might want to take a look at [Torch's `topk` function](https://docs.pytorch.org/docs/2.14/generated/torch.topk.html).

In [21]:
import torch

def search_dense(query):
    """Find the nearest neighbors in `df` for a query string, using dense embeddings.

    Arguments:
      query (str): A query string.

    Returns:
      The 10 apps (with name and description) most similar (in terms of
      cosine similarity) to the given query as a Pandas DataFrame.
    """
    # Embed query in dense representation:
    query_emb = model.encode(query, prompt_name="SearchQuery")

    # Calculate cosine similarity between query and embeddings for all documents:
    sim_tens = model.similarity(query_emb, doc_embs)

    # Pick the indeces of the top 10 apps with the highest similarity:
    _, top_inds = torch.topk(sim_tens, 10)
    return df.iloc[top_inds.flatten()]

### 🤞 Test your code

Test your implementation by running the following cell, which will sanity-check your return value and show the 10 best search results for the query _"pile up pancakes"_:

In [22]:
"""Check that searching for "pile up pancakes" returns a DataFrame with ten results,
   and that the top result is "Pancake Tower"."""

result = search_dense('pile up pancakes')
display(result)

with check_solutions() as check:
    check(isinstance(result, pd.DataFrame), "search_dense() function should return a Pandas DataFrame")
    check.equals(len(result), 10, "search_dense() function should return 10 search results")
    check.equals(result.iloc[0]["name"], "Pancake Tower", "The top search result should be 'Pancake Tower'")

,name,description
1032,Pancake Tower,Let's see how many pancakes you can pile up!!
1185,Sandwich!,"Make delicious sandwiches by folding toppings on top of each other, and serve it on a platter to eat!\n\nIn this new, delicious puzzle game it’s your job to combine bread with lettuce, tomato, salmon, cheese, onion, and many more toppings. Unlock more ingredients as you go along and the levels get harder! Are you a true sandwich master?\n\nFull of tricky levels, this game is set to eradicate your boredom. Oddly satisfying and deliciously entertaining! How high can you stack your sandwich? Re..."
343,Crowd City,Become the biggest crowd in town!\nGather people accross the city and crush your opponents with your overwhelming leadership!
544,Flip Dunk,Tap to jump and then hold your finger while in the air to flip.
175,Block Puzzle Infinite,"Drag the blocks on the bottom board to the checkerboard, try to make more full rows or columns to get score, you will lose the game if no spaces to put the whole block, having fun!"
975,My Virtual Love,New stunning application!
465,Drop Stack Ball - Fall Helix Blast Crash 3D,"The most addictive stack fall game with optimised performance for all devices!\nHelix Stack Ball is a super fun and addictive one touch casual game.\nPress and hold on the screen and let the ball go down without touching the obstacles! Hold as long as possible to make combo and break the black blocks. Let the ball fall down from the helix stacks.\nStack Ball is a 3d arcade game where players smash, bump and bounce through revolving helix platforms to reach the end.\nStack Blast Ball is a Bra..."
676,Hoop Stars,Dunk a lot! You have never played sport like this before! Feel the fun of true “reverse” basketball dribbling. Challenge the world and collect all the trophies!\nDunk a lot! You have never played sport like this before! Feel the fun of true “reverse” basketball dribbling. Challenge the world and collect all the trophies!
326,Cooking School: Games for Girls,"Children like to help their parents. They especially like to help with cooking . When there is a cooking in the kitchen, it is no way to play. But cooking is a complicated process and often it ends up with a huge mess in the kitchen. But what if you are so eager to cook pancakes, cake or cupcakes? How to cook all that without doing a cleaning after? We have a solution! Home Cooking School with our curious Hippo has opened especially for parents and children! We do not only cook food here. We..."
391,Dinosaur Rampage,Dinosaurs have been resurrected in the present age!\n\nMove your dinosaur with simple controls. Win by eating the most people and causing the most destruction! Collect coins and make your dinosaur stronger!\n\nCan you destroy the highest tower?


In [23]:
# Example — compare with the tf–idf results from Problem 3!
search_dense("puzzle adults")

,name,description
725,Jigsaw Puzzle Crown - Classic Jigsaw Puzzles,"Relax, enjoy and train your brain with Puzzle Crown, a classic jigsaw puzzle game with 1000's of high quality picture puzzles in variety of different collections suitable for every taste and preference.\n\nPuzzle Crown offers lots and lots of premium and free jigsaw puzzle packs, daily free gift puzzles and you can even build your own puzzles from family photos and images. \n\nEvery piece of the our jigsaw puzzle is unique and is individually crafted and designed to ensure a authentic and pl..."
1162,Rope Slash,"Use your finger to cut the rope and release the ball. Smash all cans to complete a level!\n\nEnjoy hundreds of handmade levels of this simple and fun arcade puzzle, wherever you are!\n\nMore levels coming soon!"
244,Can you escape the 100 room X,"Classic Escape Game ""Can you Escape the 100 room X"" is coming ...\n\nIt is a classic puzzle game , If you like the challenge must not miss it!\n\nThe new 50 room escape, Let you can't stop, start your brain cells, your observations, your judgment, your calculations, trying to escape ...\n\nHumane tips, the key moment will be pleasantly surprised,Make you successful escape.\n\nIf you love puzzle games, you will not miss such a wonderful challenge.\n\n50 rooms, 50 challenge, waiting for you to..."
173,Block Puzzle & Conquer,"- hundreds of levels, only one correct solution\n- get relax or race against the clock\n- addictive, easy and logical \n- infinite challenge\n\nThis free tangram inspired puzzle sensation is the game taking Google Play by storm. Newly updated with more features and levels, it's now even more fun! Drag the different blocks into place to match the puzzle. It’s easy to learn and free! Puzzle blocks cannot be rotated, and each level has only one correct solution. Think of it like a math problem!..."
243,Can you escape the 100 room VIII,"Classic Escape Game ""Can you Escape the 100 room VIII"" is coming ...\n\nIt is a classic puzzle game , If you like the challenge must not miss it!\n\nThe new 50 room escape, Let you can't stop, start your brain cells, your observations, your judgment, your calculations, trying to escape ...\n\nHumane tips, the key moment will be pleasantly surprised,Make you successful escape.\n\nIf you love puzzle games, you will not miss such a wonderful challenge.\n\n50 rooms, 50 challenge, waiting for you..."
533,Find the differences 750 + levels,Enjoy amazing pictures and have fun while you try to find the differences between them! \nImprove your concentration and observation skills playing Find the differences 750 + levels.\n\nSuitable for children and adults.\n\n• More than 500... even more than 750 levels all unlocked! \n• Zoom\n• Automatically saved differences\n• Portrait and landscape mode available for tablets and smartphones.\n• High Definition (HD) pictures.\n• Unlimited hints!\n• No time limit.\n\nFind the differences 750 ...
1105,Puzzle Game,"This is the most interesting block game you have ever held in your hands!\nIt gets all the charm of the puzzle game: simple, fun, attractive and suitable to everybody.\nIt is also the funniest way to train your brain and keep your mind sharp!\n\n>How to play?\nPlace the pieces into board. Once you fill in a vertical or horizontal line, it will disappear, freeing up space for new pieces.\nGame will be over if there are no space for any the given blocks below the board.\n\n>Highlights:\nA rank..."
184,Bob - jigsaw puzzles free games for kids & parents,"Free jigsaw puzzles for kids, hundreds of puzzles for toddlers to assemble. Try now kids puzzle games for toddlers.\nJigsaw puzzles are great game for your toddler to play in waiting room or anywhere while you have to wait.\n\nFeatures:\n- kid puzzle game for free\n- unlimited number of pictures, many colorful pictures to choose from by your children\n- Perfect kid game when you are waiting in line with your children\n- 4 to 100 puzzles, various difficulty levels of jigsaw puz

In [24]:
search("puzzle adults")

,name,description
1106,Puzzle Game,"Puzzle Game is a most addictive puzzle games collection, With beautiful graphics and interesting levels.\n\nPuzzle Game brings many classic fun puzzle games together，like Block Puzzle, Bubble Shooter and Onet Puzzle.\nThese puzzle games are classic and fun，and they are easy to learn.\nFrom now on, as a puzzle game lover, you don't have to search a lot of games, you can play these classic games here.\n\nThe puzzle games are as follows:\n\nBlock Puzzle\n\nBlock Puzzle is a classic puzzle that ..."
748,Kids Animals Jigsaw Puzzles ❤️🦄,"If your preschool kids like jigsaw puzzles, they will LOVE Super Puzzle!\n\nSuper Puzzle works almost like a real jigsaw puzzle for kids. When you select a puzzle piece it stays on the board even if you place it incorrectly, and you can move the puzzle piece until it slides to the correct place. \n\nEach relaxing puzzle features a beautiful image drawn by a professional artist and a unique reward when the jigsaw puzzle is completed. The images include things like unicorns, dragons, or dinosa..."
549,Food Puzzle for Kids: Preschool,"Do your kids love foods and puzzles and looking puzzle games for kids? Look no further. Food Puzzle for Kids is a fun animated puzzle game for toddlers including those with autism, preschoolers, and children from ages 1 to 6. These puzzle games are great games for kids. Your children will love kids puzzles! Kids puzzle games are a great way to spend time. We can define this toddler puzzles as the best kids puzzles 3 and 4 years old. They learn and practice with this puzzle for kids and kids ..."
168,Block Puzzle,"Simple but addictive puzzle game!\n\n""Block Puzzle"" is a fun and classic block game!\nAnd our Block Puzzle is not just that. It is more Simple and addictive! Once you start, you will not stop playing. Just have a try, you will love it!\n\nHOW TO PLAY BLOCK PUZZLE?\n1. Simply drag the blocks to move them.\n2. Try to create full lines on the grid vertically or horizontally.\n3. Blocks can't be rotated.\n4. No time limits.\n\nWHY CHOOSE THIS BLOCK PUZZLE?\n★ Exquisite Game Interface!\n★ Easy to..."
59,Animal Tower,Let's try to get animals as tall as you can!\nAnyone from adults to a children can enjoy this game.\nThere are many cute animals.
1505,Wood Block Puzzle,"Easy to play but challenging puzzle game!\n\nWood block puzzle is a wood style block game. Easy to play, but difficult to be a master. More wood blocks crush, more score you will get. Try it and you will love this block puzzle game.\n\nHow to play block puzzle:\n- Place the pieces into board. Once you fill in a vertical or horizontal line, it will disappear, freeing up space for new pieces.\n- Game will be over if there are no space for any the given blocks below the board.\n\nFeatures of th..."
174,Block Puzzle & Jigsaw Puzzle 2019,"Block Puzzle & Jigsaw Puzzle 2019 is the most addictive free game with the beautiful design and color. It's a classic combo game for puzzlemaker which is easy to play and pleasurable game for all ages.\n\nSo how to play it?\n\nSimply drag the colour blocks, and fill up all grids. If you fill one line, the block will be destroyed regardless of vertical or horizontal. Fill Wood Block Puzzle can help you train your brain and keep you mentally fit everyday.\n\nDespite that Block Puzzle & Jigsaw ..."
1517,Word Puzzle Music Box: Scramble Words Games,"Word Puzzle Music Box: Scramble Words Games is an incredible crossword puzzle game which can inspire your passion for brain challenges.\n\nDOWNLOAD to TRAIN your brain!Play it offline any time! \n\nJoin Word Puzzle Music Box and experience the tremendous brain challenging fun. Enjoy this modern Scramble Words Games with the best of word searching, anagrams and crosswords! \n\nWord Puzzle Music Box: Scramble Words Games is very simple to play, the main goal of this crossword puzzle is using t..."
167,Block Gems: Classic Block Puzzle Games,"Relax and unlock your brain with the simple but addictive pu

## Problem 5: Evaluating search results

So far, we have only judged the quality of our search engines by looking at the results for a few example queries.  To compare the two methods more systematically, we need a **gold-standard relevance judgement**: a set of queries together with information about which documents are relevant for each query.  We have prepared such judgements for ten queries in the file `relevance.tsv`.  The file only lists apps that were judged _relevant_; every other app is assumed to be non-relevant for that query.

Each row of the file contains a query, the ID of a relevant app (corresponding to the index of our DataFrame `df`), and the name of the app (for readability only):

In [25]:
relevance = pd.read_csv('relevance.tsv', sep='\t')
relevance.head(15)

,query,doc_id,name
0,pile up pancakes,326,Cooking School: Games for Girls
1,pile up pancakes,1032,Pancake Tower
2,dodge trains,1300,Subway Princess Runner
3,dodge trains,1301,Subway Surfers
4,learn a new language,64,Animals names and sounds
5,learn a new language,800,Learn English Sentence Master Pro
6,zombie survival shooter,353,DEAD TARGET: Zombie Offline
7,zombie survival shooter,354,DEAD WARFARE: Zombie Shooting - Gun Games Free
8,zombie survival shooter,805,Left to Survive: Zombie PvP Shooter
9,zombie survival shooter,847,MAD ZOMBIES : Offline Zombie Games


For evaluation, it is more convenient to have the judgements as a dictionary that maps each query to the set of relevant app IDs:

In [26]:
gold = relevance.groupby('query')['doc_id'].agg(set).to_dict()

# Number of relevant documents per query
{query: len(relevant) for query, relevant in gold.items()}

{'alphabet games for toddlers': 7,
 'board game adaptation': 7,
 'build and manage a farm': 12,
 'chess': 4,
 'dodge trains': 2,
 'home workout without equipment': 3,
 'learn a new language': 2,
 'pile up pancakes': 2,
 'relaxing sleep sounds': 2,
 'zombie survival shooter': 7}

### Precision at _k_

As discussed in the lecture, recall is hard to measure for ranked retrieval, since it requires us to know _all_ relevant documents.  Evaluation therefore tends to focus on precision.  The simplest measure is **precision at _k_**: the fraction of the top _k_ results that are relevant.

<div class="alert alert-info">

Implement a function that **computes precision at _k_** for a single ranking.

</div>

In [27]:
def precision_at_k(ranking, relevant, k=10):
    """Compute precision at rank k.

    Arguments:
      ranking (list): Document IDs, in the order returned by the search engine.
      relevant (set): Document IDs that are relevant for the query.
      k (int): The cut-off rank.

    Returns:
      The fraction of the top k documents in the ranking that are relevant.
    """
    return sum([1 if i in relevant else 0 for i in ranking[:k]])/k

### Average precision

Precision at _k_ does not care about the _order_ of the top _k_ results.  **Average precision** (AP) does: for each rank (up to _k_) at which a relevant document was returned, we compute the precision up to that rank, and then take the average of these values.  If no relevant document was returned at all, the average precision is zero.  Averaging the AP values over all queries gives the **mean average precision** (MAP).

<div class="alert alert-info">

Implement a function that **computes average precision** for a single ranking, following the definition from the lecture.

</div>

In [28]:
def average_precision(ranking, relevant, k=10):
    """Compute average precision up to rank k.

    Arguments:
      ranking (list): Document IDs, in the order returned by the search engine.
      relevant (set): Document IDs that are relevant for the query.
      k (int): The maximal rank to consider.

    Returns:
      The average of the precision values at each rank (up to k) where
      a relevant document was returned, or 0.0 if there are no such ranks.
    """
    precisions = []
    relevant_found = 0
    for rank, document_id in enumerate(ranking[:k], start=1):
        if document_id in relevant:
            relevant_found += 1
            precisions.append(relevant_found / rank)
    return sum(precisions) / len(precisions) if precisions else 0.0

### 🤞 Test your code

Test your implementation by running the following cell, which uses a toy example similar to the "best" and "worse" example rankings from the lecture:

In [29]:
"""Check precision_at_k() and average_precision() on toy examples."""

relevant = {1, 2, 3, 4}
best = [1, 2, 3, 4, 5, 6, 7, 8]
worse = [1, 5, 2, 6, 3, 7, 4, 8]

with check_solutions() as check:
    check.equals(precision_at_k(best, relevant, k=4), 1.0)
    check.equals(precision_at_k(worse, relevant, k=4), 0.5)
    check.equals(precision_at_k(worse, relevant, k=8), 0.5)
    check.isclose(average_precision(best, relevant), 1.0)
    check.isclose(average_precision(worse, relevant), 0.7095238095)

### Comparing the two search engines

We can now evaluate our two search functions.  Recall that both `search()` and `search_dense()` return a DataFrame whose index contains the IDs of the returned apps — in the order of the ranking.

<div class="alert alert-info">

Implement a function that runs a given search function on all queries in `gold`, and returns a DataFrame with the precision at 10 and the average precision for each query.

</div>

In [30]:
gold

{'alphabet games for toddlers': {21, 22, 23, 484, 759, 802, 803},
 'board game adaptation': {272, 406, 865, 1039, 1324, 1358, 1383},
 'build and manage a farm': {178,
  276,
  491,
  519,
  520,
  522,
  574,
  765,
  1367,
  1401,
  1418,
  1419},
 'chess': {88, 279, 280, 281},
 'dodge trains': {1300, 1301},
 'home workout without equipment': {540, 1194, 1534},
 'learn a new language': {64, 800},
 'pile up pancakes': {326, 1032},
 'relaxing sleep sounds': {115, 1218},
 'zombie survival shooter': {353, 354, 805, 847, 1553, 1556, 1558}}

In [31]:
def evaluate(search_fn):
    """Evaluate a search function on the gold-standard queries.

    Arguments:
      search_fn (function): A function that takes a query string and returns
        the search results as a Pandas DataFrame (like `search` or `search_dense`).

    Returns:
      A Pandas DataFrame with one row per query (used as the index) and two
      columns, 'P@10' and 'AP'.
    """
    rows = []
    for q in gold.keys():
        inds = list(search_fn(q).index)
        p10, ap = precision_at_k(ranking=inds, relevant=gold[q]), average_precision(ranking=inds, relevant=gold[q])
        rows.append({"query": q, "P@10": p10, "AP": ap})

    return pd.DataFrame(rows).set_index("query")

### 🤞 Test your code

The following cell evaluates both search engines and shows the results per query, followed by the averages over all queries (i.e., the mean P@10 and the MAP):

In [32]:
"""Check that evaluate() returns one row per query with values between 0 and 1."""

results = pd.concat({'tf–idf': evaluate(search), 'dense': evaluate(search_dense)}, axis=1)
display(results)
display(results.mean().to_frame('mean').T)

with check_solutions() as check:
    check.equals(len(results), len(gold), "evaluate() should return one row per query")
    check(set(evaluate(search).columns) == {'P@10', 'AP'}, "evaluate() should return the columns 'P@10' and 'AP'")
    check(((results >= 0) & (results <= 1)).all().all(), "All values should be between 0 and 1")

tf–idf           dense          
                                 P@10        AP  P@10        AP
query                                                          
alphabet games for toddlers       0.3  0.444444   0.7  0.924036
board game adaptation             0.5  0.686667   0.4  0.315179
build and manage a farm           0.9  0.962654   0.9  1.000000
chess                             0.4  1.000000   0.4  0.804167
dodge trains                      0.2  0.583333   0.1  1.000000
home workout without equipment    0.3  0.805556   0.3  1.000000
learn a new language              0.2  0.833333   0.1  1.000000
pile up pancakes                  0.2  1.000000   0.2  0.611111
relaxing sleep sounds             0.2  1.000000   0.2  0.833333
zombie survival shooter           0.5  0.876667   0.7  0.957143

tf–idf           dense          
       P@10        AP  P@10        AP
mean   0.37  0.819265   0.4  0.844497

## Individual reflection

<div class="alert alert-warning">

**After you have solved the lab,** save the notebook and **work on the following part individually.**

</div>

<div class="alert alert-info">

**Write a _brief_ reflection on the questions below.**

- You may discuss the questions with your lab partner, but remember to write them up individually.  **Do not copy** answers from your lab partner.
- You may add more cells below (even code cells!) if you wish.

</div>

1. In Problem&nbsp;1, which token had the highest tf–idf score, which the lowest?  Based on your knowledge of how tf–idf works, how would you explain this result?
2. How do you interpret the results from the evaluation in Problem&nbsp;5?  Discuss at least one concrete example query where the two methods clearly differ.

# A1:
`pancakes` had the highest tf-idf score, even though it only occurs often in the document, it is likely rare to occur in other games, as pancakes in a game description is rather unusual. So the IDF-Score is rather high, boosting its tf-idf score the the top.
That is exactly what tf-idf wants to achieve, with IDF: Judge words not only by how often they occur in the document itself, but also by how rare they are across all documents, so words that are unique stand out.

`you` had the lowest score. Which can be made sense of with the same reason. Modest TF-Score given the small amount of terms in the document, but it is quite common stopword among other documents, so its tf-idf score is dampened by its low IDF-Score. -> `you` in a query is not really informative on what document to show the user.


# A2:

While the dense embedding generally slightly outperforms the sparse embeddings using tf-idf, for `alphabet games for toddlers` the dense embeddings are substantially better. Looking at P@10, the dense embeddings finds all 7 in its top 10, while spare embeddings finds merely 3 of them.
This could make sense when looking at the app descriptions. Take `"Learn to Read with Tommy Turtle"` it description does not use the word `game` or `toddler` but its dense embedding incorporates the fact that this is likely not an adult game and is made for pre-schoolers/toddlers, even though it is not explicitly said in its vocabulary. This could be why dense embeddings work better for more nuanced descriptions.

In [45]:
toddler_games = list(gold['alphabet games for toddlers'])
print(df.loc[toddler_games[0], "description"])

Reading made easy!

Already enjoyed by over five million families, Learn to Read with Tommy Turtle is a delightful game that invites preschool-aged children to blend sounds into words, read and form simple words, identify spoken words and learn word families.

Its six sections include:

★ Magic Letter Bridge: kids learn to blend letter sounds into words by moving Tommy the Turtle across the letter bridge

★ Skateboards and Helmet: children practice reading as they are asked to fit Tommy's animal friends with skateboards and helmets
.
★ Turn the Blocks: kids make words in a way that's fun by turning real-looking blocks with letters on each side until the word is spelled out.

★ Word Ball: children learn to identify spoken words by helping Tommy bat the right ball (only available in full version)

★ Word Magic: kids learn about word families as they fill the corresponding pot with potion and watch magic happen (only available in full version)

★ Rocket Words: children get to practice wor

**Congratulations on finishing this lab! 👍**

<div class="alert alert-info">

➡️ Remember that each student should submit their own version of this notebook, with their own individual answers to the reflection questions.

</div>